# Segmentación de clientes con aprendizaje no supervisado
**Dataset:** Mall Customer Segmentation Data (Kaggle) · 200 clientes · edad, género, ingreso anual (k$) y puntaje de gasto (1-100).

**Objetivo:** identificar grupos naturales de comportamiento con clustering y traducir cada segmento en un plan de acción de negocio.

**Ruta:** 1) exploración → 2) preparación → 3) elección de k → 4) modelo final y validación → 5) perfil de segmentos → 6) plan de acción.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
df = pd.read_csv('Mall_Customers.csv')
ING, GAS = 'Annual Income (k$)', 'Spending Score (1-100)'
df.head()

## 1. Exploración y calidad de datos

In [ ]:
print('Filas, columnas:', df.shape)
print('Nulos:', int(df.isna().sum().sum()), '| Duplicados:', int(df.duplicated().sum()))
print(df.Gender.value_counts().to_string())
df.drop(columns='CustomerID').describe().round(1)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
for a, c in zip(ax, ['Age', ING, GAS]):
    a.hist(df[c], bins=15, color='#4C78A8', edgecolor='white'); a.set_title(c)
plt.tight_layout(); plt.show()

Sin nulos ni duplicados. En el dibujo ingreso vs. gasto (siguiente celda) ya se aprecian agrupaciones visibles, lo que anticipa que existe estructura natural en los datos.

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.scatter(df[ING], df[GAS], s=25, color='#4C78A8', alpha=.8)
plt.xlabel(ING); plt.ylabel(GAS); plt.title('Ingreso vs. puntaje de gasto'); plt.show()

## 2. Preparación
Se segmenta por **ingreso anual y puntaje de gasto**, las dos variables que describen el comportamiento de compra. Se escalan (StandardScaler) para que ninguna domine por su rango. La edad y el género no entran al algoritmo: se usan después para **perfilar** cada grupo (en una prueba con edad incluida, la silueta máxima fue 0.43 frente a 0.56, es decir, grupos menos definidos).

In [ ]:
X = StandardScaler().fit_transform(df[[ING, GAS]])
X[:3].round(2)

## 3. Elección del número de clústeres (k)

In [ ]:
ks = range(2, 11)
inercia, sil = [], []
for k in ks:
    m = KMeans(k, n_init=20, random_state=42).fit(X)
    inercia.append(m.inertia_); sil.append(silhouette_score(X, m.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(list(ks), inercia, 'o-', color='#4C78A8'); ax[0].set_title('Método del codo (inercia)'); ax[0].set_xlabel('k')
ax[1].plot(list(ks), sil, 'o-', color='#F58518'); ax[1].set_title('Coeficiente de silueta'); ax[1].set_xlabel('k')
ax[1].axvline(5, ls='--', color='gray')
plt.tight_layout(); plt.show()
pd.DataFrame({'k': list(ks), 'inercia': np.round(inercia, 1), 'silueta': np.round(sil, 3)})

El codo se aplana alrededor de **k = 5** y la silueta alcanza su máximo ahí (≈ 0.555). Además 5 grupos son un número manejable para el negocio.

## 4. Modelo final y validación

In [ ]:
km = KMeans(5, n_init=20, random_state=42).fit(X)
df['cluster'] = km.labels_

hc = AgglomerativeClustering(5, linkage='ward').fit(X)
print('Silueta K-Means  :', round(silhouette_score(X, km.labels_), 3))
print('Silueta jerárquico:', round(silhouette_score(X, hc.labels_), 3))
print('Concordancia K-Means vs. jerárquico (ARI):', round(adjusted_rand_score(km.labels_, hc.labels_), 3))

Dos algoritmos distintos llegan a una partición casi idéntica (ARI ≈ 0.94), lo que da confianza en que los grupos son reales y no un artefacto de K-Means.

## 5. Perfil de los segmentos

In [ ]:
def nombrar(r):
    if r[ING] < 45 and r[GAS] >= 50:  return 'Jóvenes entusiastas'
    if r[ING] < 45 and r[GAS] < 50:   return 'Ahorradores de bajo ingreso'
    if r[ING] >= 70 and r[GAS] >= 50: return 'Clientes VIP'
    if r[ING] >= 70 and r[GAS] < 50:  return 'Alto poder, bajo gasto'
    return 'Clientes promedio'

cent = df.groupby('cluster')[[ING, GAS]].mean()
df['segmento'] = df.cluster.map(cent.apply(nombrar, axis=1))

perfil = df.groupby('segmento').agg(
    clientes=('CustomerID', 'count'),
    edad=('Age', 'mean'),
    ingreso_k=(ING, 'mean'),
    gasto=(GAS, 'mean'),
    pct_mujeres=('Gender', lambda s: (s == 'Female').mean() * 100)).round(1)
perfil['pct_base'] = (perfil.clientes / len(df) * 100).round(1)
perfil.sort_values('ingreso_k')

In [ ]:
colores = {'Jóvenes entusiastas': '#E45756', 'Ahorradores de bajo ingreso': '#72B7B2',
           'Clientes promedio': '#9D9DA1', 'Clientes VIP': '#4C78A8', 'Alto poder, bajo gasto': '#F58518'}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for s, g in df.groupby('segmento'):
    ax[0].scatter(g[ING], g[GAS], s=28, color=colores[s], label=f'{s} ({len(g)})', alpha=.9)
cen = df.groupby('segmento')[[ING, GAS]].mean()
ax[0].scatter(cen[ING], cen[GAS], s=140, c='black', marker='X', label='Centroides')
ax[0].set_xlabel(ING); ax[0].set_ylabel(GAS); ax[0].set_title('Segmentos por ingreso y gasto'); ax[0].legend(fontsize=8)

p = PCA(2).fit_transform(X)
for s in colores:
    m = (df.segmento == s).values
    ax[1].scatter(p[m, 0], p[m, 1], s=28, color=colores[s])
ax[1].set_title('Proyección PCA de las variables escaladas'); ax[1].set_xlabel('PC1'); ax[1].set_ylabel('PC2')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
orden = perfil.sort_values('ingreso_k').index
for a, (c, t) in zip(ax, [('edad', 'Edad promedio'), ('ingreso_k', 'Ingreso anual promedio (k$)'), ('gasto', 'Puntaje de gasto promedio')]):
    a.barh(orden, perfil.loc[orden, c], color=[colores[s] for s in orden]); a.set_title(t)
for a in ax[1:]: a.set_yticklabels([])
plt.tight_layout(); plt.show()

**Lectura de los segmentos**
- **Clientes VIP** — ingreso alto y gasto alto, edad media en torno a los 33 años.
- **Alto poder, bajo gasto** — ingreso alto pero gasto muy bajo: dinero disponible que no se está capturando.
- **Jóvenes entusiastas** — ingreso bajo y gasto alto, el grupo más joven (≈ 25 años).
- **Ahorradores de bajo ingreso** — ingreso y gasto bajos, mayor edad (≈ 45 años).
- **Clientes promedio** — el grupo más grande (≈ 40 % de la base) con ingreso y gasto en la mitad de la escala.

El género prácticamente no diferencia segmentos (≈ 46-61 % mujeres en todos), por lo que no conviene segmentar por género.

## 6. Plan de acción por segmento

In [ ]:
plan = pd.DataFrame([
 ['Clientes VIP', 'Retener y aumentar el valor de vida',
  'Programa de lealtad con niveles, acceso anticipado a lanzamientos, eventos privados, atención personalizada.',
  'Retención, frecuencia de visita, gasto por cliente'],
 ['Alto poder, bajo gasto', 'Convertir poder adquisitivo en compra',
  'Descubrir por qué no compran (encuesta corta); ofertas premium y de experiencia, comunicación por canales digitales, recomendaciones personalizadas.',
  'Tasa de conversión, aumento del puntaje de gasto'],
 ['Jóvenes entusiastas', 'Aprovechar la afinidad sin erosionar margen',
  'Promociones en redes sociales, programa de referidos, combos y financiamiento en cuotas, contenido de tendencias.',
  'Referidos, frecuencia de compra, margen por ticket'],
 ['Clientes promedio', 'Subirlos de nivel',
  'Ofertas por volumen y ventas cruzadas, puntos que escalen con la compra, comunicaciones regulares de bajo costo.',
  'Migración hacia VIP, ticket promedio'],
 ['Ahorradores de bajo ingreso', 'Mantener con bajo costo',
  'Cupones y precios de oferta, productos básicos, comunicación económica (mensajería masiva). No invertir en campañas costosas.',
  'Costo por contacto, recompra'],
], columns=['Segmento', 'Objetivo', 'Acciones', 'KPI'])
plan = plan.merge(perfil[['clientes', 'pct_base']].reset_index(), left_on='Segmento', right_on='segmento').drop(columns='segmento')
plan.rename(columns={'clientes': 'Clientes', 'pct_base': '% de la base'}, inplace=True)
pd.set_option('display.max_colwidth', None)
plan[['Segmento', 'Clientes', '% de la base', 'Objetivo', 'Acciones', 'KPI']]

## Conclusiones y limitaciones
- Hay **5 segmentos naturales** y estables (validados con dos algoritmos).
- Las mayores oportunidades están en **Alto poder, bajo gasto** (ingreso sin capturar) y en **Clientes promedio** (grupo más grande, margen de ascenso).
- El dataset es pequeño (200 clientes) y sintético-didáctico: con solo dos variables de comportamiento, en producción conviene añadir frecuencia, recencia, canal y categorías de compra (análisis RFM).
- Los segmentos deben revisarse periódicamente y validarse con pruebas A/B de las acciones.